# 2. Add supplied assumptions, reasons and unit interpretations

A saved Calliope export contains numerical and structural information, but it usually does not explain why the modeller chose a particular assumption. This tutorial compares the same saved national execution with and without the included YAML annotations. It answers competency question 5: which assumptions and reasons have been supplied for selected inputs?

**The study, future scenario, supplier, dataset history and assumption explanations in this example are invented tutorial context.** They are not reconstructed from the original authors' intentions. The kW interpretation has separately identified evidence in source comments. Adding these annotations does not create another simulation or alter the saved values.

This notebook is independent of Tutorial 1. Run it from top to bottom; outputs go into `build/notebooks/02_supplied_context/`.

In [1]:
from pathlib import Path
import json
from decimal import Decimal

import pandas as pd
from IPython.display import display
from calliomapper import Translator
from calliomapper.queries import run_query
from calliomapper.vocabulary import OC

# Show complete explanations rather than shortening them with ellipses.
pd.set_option("display.max_colwidth", None)

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "examples" / "calliope-0.7.0-glpk").is_dir()
)
EXAMPLES = REPO_ROOT / "examples" / "calliope-0.7.0-glpk"
NATIONAL = EXAMPLES / "national_scale" / "results_directory"
OUT = REPO_ROOT / "build" / "notebooks" / "02_supplied_context"
OUT.mkdir(parents=True, exist_ok=True)


def answer_table(result):
    """Keep RDF numeric values and represent unavailable values as None."""
    return pd.DataFrame(
        [[term.toPython() if term is not None else None for term in row] for row in result],
        columns=[str(variable) for variable in result.vars],
    )


import yaml
from rdflib import RDF, RDFS, URIRef
from calliomapper.vocabulary import OEO, DCT

ANNOTATIONS = EXAMPLES / "annotations" / "national.yaml"
document = yaml.safe_load(ANNOTATIONS.read_text())

## Establish what the saved files can answer alone

We first export structure and capacities without supplied context. An empty assumption query means that no assumption statement has been supplied. It is not a conclusion that no assumption existed in the original modelling process.

In [2]:
source_only = Translator(NATIONAL, groups={"structure", "capacities"})
source_path = OUT / "source-only.nq"
source_only.save(source_path)
source_graph = source_only.translate()
source_assumptions = answer_table(run_query(source_graph, "cq05"))
assert source_assumptions.empty
display(pd.DataFrame([{"information available": "saved CSV files only", "CQ-05 answers": len(source_assumptions)}]))

,information available,CQ-05 answers
0,saved CSV files only,0


## Inspect the annotations before using them

The complete [annotation document](../examples/calliope-0.7.0-glpk/annotations/national.yaml) declares people and sources, then connects an assumption to a specific input. The excerpt below shows the first assumption and its reason. `targets` identifies the input by variable, node and technology; the referenced supplier, scenario and evidence are declared elsewhere in the same document.

The second assumption has no supplied reason. The graph must retain that absence rather than fill it with a plausible explanation.

In [3]:
excerpt = {
    "default_supplier": document["default_supplier"],
    "default_evidence_status": document["default_evidence_status"],
    "assumptions": [document["assumptions"][0]],
}
print(yaml.safe_dump(excerpt, sort_keys=False))
assert "reasons" not in document["assumptions"][1]

default_supplier: demonstration_contributor
default_evidence_status: Synthetic supplied context for this tutorial; not an empirical
  claim.
assumptions:
- id: capacity_assumption
  statement: In this invented future, the region1 CCGT capacity is limited to 30000
    kW.
  scenario: reference_future
  targets:
  - group: inputs
    variable: flow_cap_max
    node: region1
    technology: ccgt
  evidence:
  - invented_report
  reasons:
  - id: capacity_reason
    statement: The fictional contributor chose this limit to demonstrate a capacity
      restriction. It is not an explanation supplied by the example authors.
    evidence:
    - invented_report



## Add the supplied context and query the resulting graph

The identifier manifest is reused so that the source-only and annotated exports identify the same model components and execution. The new statements identify who supplied them and their evidence status, and are kept in a separate named graph. Validation checks their structure and consistency with the selected records; it does not establish that the claims are scientifically correct.

In [4]:
annotated = Translator(
    NATIONAL, groups={"structure", "capacities"}, annotations=ANNOTATIONS,
    identity_file=source_path.with_suffix(".nq.manifest.json"),
)
annotated.save(OUT / "annotated.nq")
annotated_graph = annotated.translate()
assumptions = answer_table(run_query(annotated_graph, "cq05"))

assert len(assumptions) == 2
assert assumptions["reasonText"].notna().sum() == 1
assert annotated.context.execution == source_only.context.execution
assumptions["supplier"] = assumptions["assumptionSupplier"].map(
    lambda identifier: str(annotated_graph.value(URIRef(identifier), RDFS.label))
)
display(assumptions[["statement", "reasonText", "supplier"]].fillna("Not supplied"))

,statement,reasonText,supplier
0,"In this invented future, the region1 CCGT capacity is limited to 30000 kW.",The fictional contributor chose this limit to demonstrate a capacity restriction. It is not an explanation supplied by the example authors.,Fictional tutorial contributor
1,The saved battery charging efficiency is retained in this illustrative future.,Not supplied,Fictional tutorial contributor


## Inspect a unit interpretation and its evidence

The saved number `30000` does not automatically become `30000 kW`. The supplied interpretation identifies the target record, the OEO class for kilowatt, the label, and the evidence supporting that interpretation. Here the evidence is the original example's kW comment and the saved capacity-bound mathematics. This remains an attributed interpretation rather than a unit field found in the CSV.

In [5]:
print(yaml.safe_dump(document["units"][0], sort_keys=False))
unit_evidence = []
for quantity in annotated_graph.subjects(RDF.type, OEO.OEO_00000350):
    unit = annotated_graph.value(quantity, OEO.OEO_00040010)
    supplier = annotated_graph.value(quantity, OC.suppliedBy)
    sources = [str(annotated_graph.value(source, DCT.title))
               for source in annotated_graph.objects(quantity, DCT.source)]
    unit_evidence.append({
        "interpretation": str(annotated_graph.value(quantity, DCT.identifier)),
        "unit": str(annotated_graph.value(unit, RDFS.label)),
        "supplier": str(annotated_graph.value(supplier, RDFS.label)),
        "evidence source": "; ".join(sources),
    })
assert len(unit_evidence) == 2
display(pd.DataFrame(unit_evidence))

id: bound_unit_interpretation
target:
  group: inputs
  variable: flow_cap_max
  node: region1
  technology: ccgt
unit_class: https://openenergyplatform.org/ontology/oeo/OEO_00390000
label: kW
evidence:
- capacity_comment
evidence_status: Interpretation grounded in the original example's kW comment and
  the saved capacity bound, supplied here by the tutorial contributor.



,interpretation,unit,supplier,evidence source
0,result_unit_interpretation,kW,Fictional tutorial contributor,Calliope national example capacity-unit comment and mathematical bound
1,bound_unit_interpretation,kW,Fictional tutorial contributor,Calliope national example capacity-unit comment and mathematical bound


## Distinguish the report author from the annotation contributor and supplier

Suppose a report was written by one researcher, a second researcher wrote an annotation explaining how to use it, and a third person supplied that annotation for this export. These are three different statements. The first two use OEO's author and contributor relationships. The third uses OntoCal's supplier relationship.

The next cell adds **explicitly invented tutorial facts** to a copy of the annotation document. It does not discover authorship from the Calliope files. We declare a fictional report author and annotation contributor, while keeping the original supplier. The assertions check that supplying the annotation did not turn that person into the report's author, and that contributing to the annotation did not turn its contributor into the report's author.

In [6]:
from copy import deepcopy

with_roles = deepcopy(document)
with_roles["agents"].extend([
    {"id": "report_author", "kind": "person", "label": "Fictional author of the original report"},
    {"id": "annotation_writer", "kind": "person", "label": "Fictional contributor to the new annotation"},
])
with_roles["sources"][0]["authors"] = ["report_author"]
with_roles["assumptions"][0]["reasons"][0]["contributor"] = "annotation_writer"
role_file = OUT / "annotations-with-explicit-roles.yaml"
role_file.write_text(yaml.safe_dump(with_roles, sort_keys=False))
role_export = Translator(NATIONAL, groups={"structure", "capacities"}, annotations=role_file)
role_graph = role_export.translate()
report = next(subject for subject, title in role_graph.subject_objects(DCT.title)
              if str(title) == with_roles["sources"][0]["title"])
reason = next(subject for subject, text in role_graph.subject_objects(RDFS.comment)
              if str(text).strip() == with_roles["assumptions"][0]["reasons"][0]["statement"].strip())
author = role_graph.value(report, OEO.OEO_00000506)
contributor = role_graph.value(reason, OEO.OEO_00140008)
supplier = role_graph.value(reason, OC.suppliedBy)
assert len({author, contributor, supplier}) == 3
assert (report, OEO.OEO_00000506, contributor) not in role_graph
assert (report, OEO.OEO_00000506, supplier) not in role_graph
display(pd.DataFrame([
    {"record": "Original report", "relationship": "author", "person": str(role_graph.value(author, RDFS.label))},
    {"record": "New annotation", "relationship": "contributor", "person": str(role_graph.value(contributor, RDFS.label))},
    {"record": "New annotation", "relationship": "supplier", "person": str(role_graph.value(supplier, RDFS.label))},
]))

,record,relationship,person
0,Original report,author,Fictional author of the original report
1,New annotation,contributor,Fictional contributor to the new annotation
2,New annotation,supplier,Fictional tutorial contributor


## Confirm that annotations add meaning without changing the numbers

Both exports contain the same selected capacity bound and calculated capacity. Only the annotated graph supplies the supported kW interpretation. The table below displays the two versions together; their identifiers remain available in the full query results.

In [7]:
capacity_views = []
for label, graph in [("saved files only", source_graph), ("with supplied annotations", annotated_graph)]:
    rows = answer_table(run_query(graph, "cq03", bindings={"node": "region1", "technology": "ccgt"}))
    rows = rows.loc[rows["variable"].isin(["flow_cap_max", "flow_cap"])].copy()
    assert len(rows) == 2
    assert all(number == Decimal("30000.0") for number in rows["value"])
    if label == "saved files only":
        assert rows["unit"].isna().all()
    else:
        assert (rows["unit"] == "kW").all()
    rows.insert(0, "information available", label)
    capacity_views.append(rows[["information available", "role", "variable", "value", "unit"]])
display(pd.concat(capacity_views, ignore_index=True))

,information available,role,variable,value,unit
0,saved files only,result,flow_cap,30000.0,None
1,saved files only,input,flow_cap_max,30000.0,None
2,with supplied annotations,result,flow_cap,30000.0,kW
3,with supplied annotations,input,flow_cap_max,30000.0,kW


To prepare your own annotations, copy the YAML file to your working directory and replace its fictional content, people and sources with statements you can support. Do not reuse the tutorial's invented provenance as evidence for a real model. The [annotation guide](../examples/annotations/README.md) describes the supported fields and required evidence. [Tutorial 3](03_comparison_and_integration.ipynb) shows the additional information needed to compare separately identified exports and connect an external catalogue.